In [3]:
import pandas as pd
#Structuring our Data
#Load your CSV file
og_df = pd.read_csv("Structured Data copy.csv")

# Reshape the data from wide to long format
# We use 'id_vars' for columns to keep, and everything else (dates) becomes rows
df = pd.melt( 
    og_df,
    id_vars=['COUNTRY', 'VARIABLE ', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY', 'SCALE'],
    var_name='Date',
    value_name='Value'
)

In [4]:
df.head() 

,COUNTRY,VARIABLE,TYPE_OF_TRANSFORMATION,FREQUENCY,SCALE,Date,Value
0,"Gambia, The","Consumer price index (CPI) - Housing, water, e...",Index,Monthly,Units,2012-M01,61.006604
1,"Gambia, The","Consumer price index (CPI) - Housing, water, e...","Period average, Year-over-year (YOY) percent c...",Monthly,Units,2012-M01,4.918362
2,"Gambia, The","Consumer price index (CPI) - Housing, water, e...",Weight,Monthly,Units,2012-M01,33.730562
3,"Gambia, The",Consumer price index (CPI) - Food and non-alco...,Weight,Monthly,Units,2012-M01,547.362054
4,"Gambia, The",Consumer price index (CPI) - Food and non-alco...,Index,Monthly,Units,2012-M01,56.876508


In [5]:
# Strip accidental trailing/leading whitespaces from column names and string metrics
df.columns = df.columns.str.strip()
df['COUNTRY'] = df['COUNTRY'].str.strip()
df['VARIABLE'] = df['VARIABLE'].str.strip()

# Standardize text dates (e.g., '2012-M01') into true system Date objects ('2012-01-01')
df['Date'] = pd.to_datetime(df['Date'].str.replace('-M', '-'), format='%Y-%m')

# Save the perfectly structured file
df.to_csv("Final_Structured_Data.csv", index=False)


In [6]:
#Preparing the structured data
df = pd.read_csv("Final_Structured_Data.csv")

df.head(20)

,COUNTRY,VARIABLE,TYPE_OF_TRANSFORMATION,FREQUENCY,SCALE,Date,Value
0,"Gambia, The","Consumer price index (CPI) - Housing, water, e...",Index,Monthly,Units,2012-01-01,61.006604
1,"Gambia, The","Consumer price index (CPI) - Housing, water, e...","Period average, Year-over-year (YOY) percent c...",Monthly,Units,2012-01-01,4.918362
2,"Gambia, The","Consumer price index (CPI) - Housing, water, e...",Weight,Monthly,Units,2012-01-01,33.730562
3,"Gambia, The",Consumer price index (CPI) - Food and non-alco...,Weight,Monthly,Units,2012-01-01,547.362054
4,"Gambia, The",Consumer price index (CPI) - Food and non-alco...,Index,Monthly,Units,2012-01-01,56.876508
5,"Gambia, The",Consumer price index (CPI) - Food and non-alco...,"Period average, Year-over-year (YOY) percent c...",Monthly,Units,2012-01-01,5.456840
6,"Gambia, The",Consumer price index (CPI) - Transport,Index,Monthly,Units,2012-01-01,64.511532
7,"Gambia, The",Consumer price index (CPI) - Transport,"Period average, Year-over-year (YOY) percent c...",Monthly,Units,2012-01-01,6.151638
8,"Gambia, The",Consumer price index (CPI) - Transport,Weight,Monthly,Units,2012-01-01,43.681439
9,"Gambia, The",Exchage Rate - Domestic currency per US Dollar,End-of-period (EoP),Monthly,Units,2012-01-01,30.500000


In [8]:
df['Date'] = pd.to_datetime(df['Date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20862 entries, 0 to 20861
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   COUNTRY                 20862 non-null  object        
 1   VARIABLE                20862 non-null  object        
 2   TYPE_OF_TRANSFORMATION  16245 non-null  object        
 3   FREQUENCY               20691 non-null  object        
 4   SCALE                   20691 non-null  object        
 5   Date                    20862 non-null  datetime64[ns]
 6   Value                   16552 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 1.1+ MB


In [9]:
# Isolate Gambia - If you feed it a giant melting pot of data from different countries, the distinct economic patterns of smaller nations 
#get completely drowned out by the patterns of larger economies. By filtering the data to only rows where COUNTRY == 'Gambia, The',
#you create a specialized, custom-tailored environment.

gambia = df[df['COUNTRY'] == 'Gambia, The'].copy()

gambia['VARIABLE'] = gambia['VARIABLE'].str.strip()
gambia['TYPE_OF_TRANSFORMATION'] = gambia['TYPE_OF_TRANSFORMATION'].str.strip().fillna('')
gambia['Feature_Name'] = gambia['VARIABLE'] + " (" + gambia['TYPE_OF_TRANSFORMATION'] + ")"


In [10]:
# 3. Pivot the dataset into standard wide feature columns
#pivoting transforms the structural meaning of your table columns:
#Before Pivoting (Long): Columns represent metadata tags (COUNTRY, VARIABLE, VALUE). 
#The actual economic concepts are trapped as text sentences inside a column.
#After Pivoting (Wide): Columns represent distinct mathematical dimensions (Exchange_Rate, TBill_91D, Inflation_Food).
#By turning text labels into independent columns, you give the AI model a clear multidimensional map to calculate exactly 
#how a wiggle in the Exchange_Rate column creates a corresponding wiggle in the Inflation_Food column.

gambia_matrix = gambia.pivot(index='Date', columns='Feature_Name', values='Value')

columns_mapping = {
    'Consumer price index (CPI) - Food and non-alcoholic beverages (Index)': 'CPI_Food',
    'Consumer price index (CPI) - Housing, water, electricity, gas and other fuels (Index)': 'CPI_Housing',
    'Consumer price index (CPI) - Transport (Index)': 'CPI_Transport',
    'Exchage Rate -Domestic currency per US Dollar (Period average)': 'Exchange_Rate',
    'Interest Rates - Monetary policy-related, Rate, Percent per annum ()': 'Policy_Rate',
    'Interest Rates - Government securities: Treasury bills yields, Rate, Percent per annum ()': 'TBill_91D'
}
final_gambia = gambia_matrix[list(columns_mapping.keys())].rename(columns=columns_mapping)
final_gambia = final_gambia.interpolate(method='linear', limit_direction='both')


In [15]:
#The first 12 rows of your dataset (the year 2012) are removed because they do not have a prior year to calculate YoY inflation against.
#The remaining rows from 2013 onwards are completely full, clean, transformed, and ready to be loaded into our AI algorithms

# A. Food Inflation (YoY percentage change of food index)
final_gambia['Food_Inflation'] = final_gambia['CPI_Food'].pct_change(periods=12, fill_method=None) * 100

# B. General Inflation (Composite index calculated across all tracked sectors)
final_gambia['CPI_Total'] = final_gambia[['CPI_Food', 'CPI_Housing', 'CPI_Transport']].mean(axis=1)
final_gambia['Inflation'] = final_gambia['CPI_Total'].pct_change(periods=12, fill_method=None) * 100

# C. Inflation_Lag1 (Shift backwards by 1 step so previous month's inflation sits in current row)
final_gambia['Inflation_Lag1'] = final_gambia['Inflation'].shift(1)

# D. Exchange_Change (Month-over-Month percentage momentum of the Dalasi)
final_gambia['Exchange_Change'] = final_gambia['Exchange_Rate'].pct_change(periods=1, fill_method=None) * 100

# E. Add Lag Variables - Inflation forecasting depends heavily on past values
final_gambia['Exchange_Change_Lag1'] = final_gambia['Exchange_Change'].shift(1)


In [ ]:
Separating inflation into General Inflation, Food Inflation, and an Inflation Lag is a strategic modeling choice. 
In macroeconomics and machine learning, treating inflation as a single, uniform number hides the actual drivers of price changes.

why each one is necessary for your model to make accurate 1-to-3 month predictions in The Gambia.

Food inflation measures price changes specifically for food and non-alcoholic beverages.
* Why isolate it? In developing economies like The Gambia, food makes up a massive percentage of the average household's total monthly spending.
* Model Purpose: Isolating this column allows you to set it up as your specific Target variable ($y$).

General inflation represents the average price change across all tracked sectors combined (Food, Housing, and Transport).
* Why isolate it? General inflation gives the model a birds-eye view of the entire economy. If General Inflation is high
* Model Purpose: This acts as a Feature (Input clue).

Inflation Lag 1 (The Autoregressive Anchor)
An inflation lag is simply the previous month’s inflation rate pulled forward into the current row.
* Why isolate it? Inflation is highly autoregressive, meaning the single best predictor of what inflation will look like tomorrow is 
what inflation looks like today

* Model Purpose: This acts as the model's Anchor feature. Without a lag feature, if you only give an AI model interest rates and 
exchange rates, it has to guess the absolute inflation number from scratch, which leads to wildly inaccurate predictions. 
Giving it Inflation_Lag1 provides a starting baseline

In [18]:
# SUBSET TO SPECIFIED COLUMNS & CLEAN NA ENTRIES
# (Oil prices are omitted since they aren't part of your uploaded dataset variables)

requested_columns = ['Inflation', 'Inflation_Lag1', 'Food_Inflation', 'Exchange_Change','Exchange_Change_Lag1', 'Policy_Rate', 'TBill_91D']
output_df = final_gambia[requested_columns].dropna().reset_index()

output_df.head(20)

Feature_Name,Date,Inflation,Inflation_Lag1,Food_Inflation,Exchange_Change,Exchange_Change_Lag1,Policy_Rate,TBill_91D
0,2013-02-01,12.120296,11.814164,6.175472,-0.418346,0.420103,12.0,12.144490
1,2013-03-01,12.149440,12.120296,6.260989,2.060200,-0.418346,12.0,12.216735
2,2013-04-01,12.990894,12.149440,6.530455,1.926987,2.060200,12.0,12.288980
3,2013-05-01,13.828194,12.990894,6.583939,3.747921,1.926987,14.0,12.361224
4,2013-06-01,14.049153,13.828194,6.676119,4.908226,3.747921,18.0,12.433469
5,2013-07-01,12.311973,14.049153,7.079580,-10.870918,4.908226,20.0,12.505714
6,2013-08-01,12.105259,12.311973,7.117053,3.225152,-10.870918,20.0,12.577959
7,2013-09-01,11.683928,12.105259,7.231465,-2.639332,3.225152,20.0,12.650204
8,2013-10-01,9.981328,11.683928,7.290714,3.400102,-2.639332,20.0,12.722449
9,2013-11-01,10.293092,9.981328,7.016537,7.955917,3.400102,20.0,12.794694


In [19]:
#  'output_df' is our current dataframe containing the columns: 
# 'Date', 'Inflation', 'Inflation_Lag1', 'Food_Inflation', 'Exchange_Change', 'Policy_Rate', 'TBill_91D'

#Create Future Target Columns for FOOD INFLATION (If you want to predict food specifically)
output_df['Target_Food_Inflation_Month1'] = output_df['Food_Inflation'].shift(-1)
output_df['Target_Food_Inflation_Month2'] = output_df['Food_Inflation'].shift(-2)
output_df['Target_Food_Inflation_Month3'] = output_df['Food_Inflation'].shift(-3)

# Verify the new columns look correct
output_df.tail(10)



Feature_Name,Date,Inflation,Inflation_Lag1,Food_Inflation,Exchange_Change,Exchange_Change_Lag1,Policy_Rate,TBill_91D,Target_Food_Inflation_Month1,Target_Food_Inflation_Month2,Target_Food_Inflation_Month3
148,2025-06-01,8.730076,8.959829,7.760216,0.417711,-0.024093,17.0,15.43,8.461797,8.381006,7.371470
149,2025-07-01,8.020002,8.730076,8.461797,0.152614,0.417711,17.0,13.48,8.381006,7.371470,6.539318
150,2025-08-01,7.873557,8.020002,8.381006,-0.034596,0.152614,17.0,11.50,7.371470,6.539318,5.546341
151,2025-09-01,5.969097,7.873557,7.371470,0.021623,-0.034596,17.0,11.76,6.539318,5.546341,4.675281
152,2025-10-01,5.170141,5.969097,6.539318,0.150402,0.021623,17.0,11.84,5.546341,4.675281,4.028840
153,2025-11-01,4.417512,5.170141,5.546341,0.123912,0.150402,17.0,16.18,4.675281,4.028840,3.418309
154,2025-12-01,3.754798,4.417512,4.675281,0.374711,0.123912,17.0,16.18,4.028840,3.418309,3.178128
155,2026-01-01,2.933188,3.754798,4.028840,0.000000,0.374711,17.0,16.18,3.418309,3.178128,NaN
156,2026-02-01,2.359477,2.933188,3.418309,0.000000,0.000000,17.0,16.18,3.178128,NaN,NaN
157,2026-03-01,2.053039,2.359477,3.178128,0.000000,0.000000,17.0,16.18,NaN,NaN,NaN


In [ ]:
the last 3 rows will contain NaN values in our new Target columns.
Because your historical data ends at a specific month, the computer cannot look 1, 2, or 3 months into the future because those
months haven't happened yet in your file.
During Training: we will drop these last 3 rows using train_df = output_df.dropna() 
because the AI cannot train on questions that don't have answers

In [20]:
#SAVE THE FINAL MACHINE LEARNING READY CSV FILE
output_df.to_csv("Processed_Data_ML.csv", index=False)

In [21]:
print("SUCCESS! our complete machine learning training file has been saved.")
print(f"Total structured months available for training: {output_df.shape[0]}")

SUCCESS! our complete machine learning training file has been saved.
Total structured months available for training: 158
